# OASIS-1 3D Benchmark GIFs

Notebook này tạo nhiều kiểu GIF để xem kiểu visualize nào hữu ích nhất. Nó đọc output benchmark đã có. Chỉ hai cell cuối rerun selected pair nhỏ để tạo GIF iteration thật cho PSO và Classical, không train lại và không rerun full benchmark.

In [ ]:
# Setup: path, data, helper functions. Cell này không tạo GIF.
from pathlib import Path
import io
import json
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from scipy.ndimage import map_coordinates

def find_repo_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    candidates += [path / 'BioMedReg' for path in [cwd, *cwd.parents]]
    for path in candidates:
        if (path / 'src').is_dir() and (path / 'outputs').is_dir():
            return path.resolve()
    raise RuntimeError('Cannot find BioMedReg repo root. Run this notebook from BioMedReg or its parent folder.')

ROOT = find_repo_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils.io import read_image
from src.utils.metrics import normalize_image
from src.methods.classical.register import run_classical
from src.methods.metaheuristic.pso import (
    _bounds,
    _identity_vector,
    _score,
    _vector_to_params,
    _warp_rigid_3d,
)

benchmark_dir = Path('outputs/benchmark/oasis1_3d_functional_20260527_025659')
gif_dir = Path('visualize/oasis3d_benchmark_gifs')
gif_dir.mkdir(parents=True, exist_ok=True)

method_order = ['classical', 'pso', 'voxelmorph', 'transmorph']
dense_methods = ['classical', 'voxelmorph', 'transmorph']
method_labels = {
    'classical': 'Classical',
    'pso': 'PSO',
    'voxelmorph': 'VoxelMorph',
    'transmorph': 'TransMorph',
}
method_colors = {
    'classical': '#4C78A8',
    'pso': '#F58518',
    'voxelmorph': '#54A24B',
    'transmorph': '#B279A2',
}
selected_label_values = [2, 3, 4, 10, 11, 12, 13, 17, 18, 26, 41, 42, 43, 49, 50, 51, 52, 53, 54, 58]

df = pd.read_csv(benchmark_dir / 'benchmark_results.csv')
df = df[df['success'].astype(bool)].copy()
for col in ['run_seconds', 'after_mse', 'delta_mse', 'after_ncc', 'delta_ncc', 'dice_after_mean', 'dice_delta_mean']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df['pair_index'] = pd.to_numeric(df['pair_index']).astype(int)
methods = [m for m in method_order if m in set(df['method'])]

pair_mean = df.groupby('pair_index').mean(numeric_only=True)
best_pair = int(pair_mean['delta_ncc'].idxmax())
worst_pair = int(pair_mean['delta_ncc'].idxmin())
median_pair = int((pair_mean['after_ncc'] - pair_mean['after_ncc'].median()).abs().idxmin())
ncc_pivot = df.pivot(index='pair_index', columns='method', values='after_ncc')
dice_pivot = df.pivot(index='pair_index', columns='method', values='dice_after_mean')
conflict_score = (ncc_pivot['voxelmorph'] - ncc_pivot['pso']) + (dice_pivot['pso'] - dice_pivot['voxelmorph'])
conflict_pair = int(conflict_score.dropna().idxmax())

selected_cases = []
for name, pair in [
    ('best_delta_ncc', best_pair),
    ('median_after_ncc', median_pair),
    ('worst_delta_ncc', worst_pair),
    ('vm_ncc_vs_pso_dice_conflict', conflict_pair),
]:
    if pair not in [item['pair'] for item in selected_cases]:
        selected_cases.append({'name': name, 'pair': pair})
demo_pair = median_pair

def row_for(pair_index, method):
    rows = df[(df['pair_index'] == pair_index) & (df['method'] == method)]
    if rows.empty:
        raise ValueError(f'No benchmark row for pair={pair_index}, method={method}')
    return rows.iloc[0]

def mid_slice(volume, axis=0, index=None):
    if index is None:
        index = volume.shape[axis] // 2
    if axis == 0:
        return volume[index, :, :]
    if axis == 1:
        return volume[:, index, :]
    return volume[:, :, index]

def as_display(image):
    return normalize_image(np.nan_to_num(image.astype(np.float32), nan=0.0))

def overlay_rgb(fixed, candidate):
    fixed_s = as_display(fixed)
    cand_s = as_display(candidate)
    rgb = np.zeros(fixed_s.shape + (3,), dtype=np.float32)
    rgb[..., 0] = fixed_s
    rgb[..., 1] = cand_s
    rgb[..., 2] = 0.25 * fixed_s
    return np.clip(rgb, 0.0, 1.0)

def dense_field_to_zyx(field, method):
    return field[..., [2, 1, 0]] if method == 'classical' else field

def coords_3d(shape):
    return np.meshgrid(
        np.arange(shape[0], dtype=np.float32),
        np.arange(shape[1], dtype=np.float32),
        np.arange(shape[2], dtype=np.float32),
        indexing='ij',
    )

def warp_dense_scaled(moving, field, method, scale):
    field_zyx = dense_field_to_zyx(field, method)
    zz, yy, xx = coords_3d(moving.shape)
    warped = map_coordinates(
        moving.astype(np.float32),
        [zz + scale * field_zyx[..., 0], yy + scale * field_zyx[..., 1], xx + scale * field_zyx[..., 2]],
        order=1,
        mode='constant',
        cval=0.0,
        prefilter=False,
    )
    return warped.astype(np.float32)

def jacobian_det_3d(field_zyx):
    gz0, gy0, gx0 = np.gradient(field_zyx[..., 0], edge_order=1)
    gz1, gy1, gx1 = np.gradient(field_zyx[..., 1], edge_order=1)
    gz2, gy2, gx2 = np.gradient(field_zyx[..., 2], edge_order=1)
    j00, j01, j02 = 1.0 + gz0, gy0, gx0
    j10, j11, j12 = gz1, 1.0 + gy1, gx1
    j20, j21, j22 = gz2, gy2, 1.0 + gx2
    return j00 * (j11 * j22 - j12 * j21) - j01 * (j10 * j22 - j12 * j20) + j02 * (j10 * j21 - j11 * j20)

def fig_to_frame(fig, dpi=105):
    buffer = io.BytesIO()
    fig.savefig(buffer, format='png', dpi=dpi, bbox_inches='tight')
    plt.close(fig)
    buffer.seek(0)
    return Image.open(buffer).convert('RGB')

def save_gif(frames, path, duration=120):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frames[0].save(path, save_all=True, append_images=frames[1:], duration=duration, loop=0, optimize=True)
    print(f'Saved {len(frames)} frames: {path}')
    return path

def registration_frame(fixed_sl, candidate_sl, title, candidate_title):
    diff = np.abs(as_display(fixed_sl) - as_display(candidate_sl))
    fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
    fig.suptitle(title, fontsize=13)
    panels = [
        (fixed_sl, 'Fixed', 'gray'),
        (candidate_sl, candidate_title, 'gray'),
        (overlay_rgb(fixed_sl, candidate_sl), 'Overlay: fixed red, candidate green', None),
        (diff, '|fixed - candidate|', 'magma'),
    ]
    for ax, (image, subtitle, cmap) in zip(axes, panels):
        ax.imshow(image, cmap=cmap) if cmap else ax.imshow(image)
        ax.set_title(subtitle, fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
    return fig_to_frame(fig)

print(f'Repo root: {ROOT}')
print(f'Benchmark: {benchmark_dir}')
print(f'GIF output: {gif_dir}')
print('Selected cases:', selected_cases)
print('Demo pair for iteration/deformation/labels:', demo_pair)

In [ ]:
# GIF type 1: before/after blink, không nội suy fake.
# Giữ filename *_fade.gif để ghi đè file cũ, nhưng nội dung giờ là moving thật <-> registered thật.
out = gif_dir / '01_moving_to_registered_fade'
n_frames = 16
for case in selected_cases:
    pair_index = case['pair']
    base = row_for(pair_index, methods[0])
    fixed = read_image(base['fixed'])
    moving = read_image(base['moving'])
    fixed_sl = mid_slice(fixed, axis=0)
    moving_sl = mid_slice(moving, axis=0)
    for method in methods:
        row = row_for(pair_index, method)
        registered = read_image(row['registered'])
        registered_sl = mid_slice(registered, axis=0)
        frames = []
        for frame_idx in range(n_frames):
            use_after = frame_idx % 2 == 1
            candidate = registered_sl if use_after else moving_sl
            state = 'AFTER: registered image' if use_after else 'BEFORE: moving image'
            title = f"{case['name']} | pair {pair_index:03d} | {method_labels[method]} | {state}"
            frames.append(registration_frame(fixed_sl, candidate, title, state))
        save_gif(frames, out / f"{case['name']}_pair_{pair_index:03d}_{method}_fade.gif", duration=520)


In [ ]:
# GIF type 2: axial slice sweep cho một pair đại diện, tất cả methods.
out = gif_dir / '02_axial_slice_sweep'
pair_index = demo_pair
base = row_for(pair_index, methods[0])
fixed = read_image(base['fixed'])
moving = read_image(base['moving'])
slice_indices = np.linspace(8, fixed.shape[0] - 9, 22).astype(int)
for method in methods:
    row = row_for(pair_index, method)
    registered = read_image(row['registered'])
    frames = []
    for z in slice_indices:
        fixed_sl = mid_slice(fixed, axis=0, index=int(z))
        registered_sl = mid_slice(registered, axis=0, index=int(z))
        title = f'Pair {pair_index:03d} | {method_labels[method]} | axial slice z={int(z)}'
        frames.append(registration_frame(fixed_sl, registered_sl, title, 'Registered'))
    save_gif(frames, out / f'pair_{pair_index:03d}_{method}_axial_sweep.gif', duration=240)


In [ ]:
# GIF type 3: deformation/Jacobian axial sweep cho dense methods.
# Giữ folder/filename cũ để ghi đè, nhưng nội dung không còn là scale fake như type 1.
out = gif_dir / '03_final_field_scale'
pair_index = demo_pair
for method in dense_methods:
    field_path = benchmark_dir / 'test_runs' / method / f'pair_{pair_index:03d}' / 'deformation_field.npy'
    field = np.load(field_path)
    field_zyx = dense_field_to_zyx(field, method)
    magnitude = np.linalg.norm(field_zyx, axis=-1)
    jac = jacobian_det_3d(field_zyx)
    fold = jac <= 0.0
    slice_indices = np.linspace(8, magnitude.shape[0] - 9, 24).astype(int)
    frames = []
    for z in slice_indices:
        fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
        fig.suptitle(f'Pair {pair_index:03d} | {method_labels[method]} | deformation diagnostics | axial z={int(z)}', fontsize=13)
        panels = [
            (magnitude[int(z)], 'Displacement magnitude', 'viridis'),
            (jac[int(z)], 'Jacobian determinant', 'coolwarm'),
            (fold[int(z)].astype(np.float32), 'Folding mask (J <= 0)', 'Reds'),
        ]
        for ax, (image, subtitle, cmap) in zip(axes, panels):
            im = ax.imshow(image, cmap=cmap)
            ax.set_title(subtitle, fontsize=10)
            ax.set_xticks([])
            ax.set_yticks([])
            fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
        frames.append(fig_to_frame(fig))
    save_gif(frames, out / f'pair_{pair_index:03d}_{method}_field_scale.gif', duration=210)


In [ ]:
# GIF type 4: label contour trước/sau trên fixed MRI.
# Red = fixed label. Yellow fades out = moving label before. Cyan fades in = warped registered label.
out = gif_dir / '04_label_contours'
pair_index = demo_pair
base = row_for(pair_index, methods[0])
fixed = read_image(base['fixed'])
fixed_label = read_image(str(base['fixed']).replace('/fixed_', '/fixed_label_'))
moving_label = read_image(str(base['moving']).replace('/moving_', '/moving_label_'))
fixed_sl = mid_slice(fixed, axis=0)
fixed_mask = np.isin(mid_slice(fixed_label, axis=0).astype(int), selected_label_values)
moving_mask = np.isin(mid_slice(moving_label, axis=0).astype(int), selected_label_values)
for method in methods:
    reg_label_path = benchmark_dir / 'test_runs' / method / f'pair_{pair_index:03d}' / 'registered_label.nii.gz'
    registered_label = read_image(reg_label_path)
    registered_mask = np.isin(mid_slice(registered_label, axis=0).astype(int), selected_label_values)
    frames = []
    for alpha in np.linspace(0.0, 1.0, 16):
        fig, ax = plt.subplots(1, 1, figsize=(5.2, 5.2))
        fig.suptitle(f'Pair {pair_index:03d} | {method_labels[method]} | label contour fade {alpha:.2f}', fontsize=12)
        ax.imshow(fixed_sl, cmap='gray')
        ax.contour(fixed_mask.astype(float), levels=[0.5], colors='red', linewidths=1.2)
        ax.contour(moving_mask.astype(float), levels=[0.5], colors='yellow', linewidths=1.0, alpha=1.0 - alpha)
        ax.contour(registered_mask.astype(float), levels=[0.5], colors='cyan', linewidths=1.0, alpha=alpha)
        ax.text(2, 6, 'fixed label: red', color='red', fontsize=9, weight='bold')
        ax.text(2, 11, 'moving before: yellow', color='yellow', fontsize=9, weight='bold', alpha=max(0.25, 1.0 - alpha))
        ax.text(2, 16, 'warped after: cyan', color='cyan', fontsize=9, weight='bold', alpha=max(0.25, alpha))
        ax.set_xticks([])
        ax.set_yticks([])
        frames.append(fig_to_frame(fig, dpi=115))
    save_gif(frames, out / f'pair_{pair_index:03d}_{method}_label_contours.gif', duration=120)


In [ ]:
# GIF type 5A: PSO iteration thật, chuẩn hóa cùng 16 frame progress.
# Biến pso_progress_volumes sẽ được cell sau dùng để tạo GIF so sánh 4 methods cùng độ dài.
out = gif_dir / '05_true_iterations'
pair_index = demo_pair
base = row_for(pair_index, 'pso')
fixed = normalize_image(read_image(base['fixed']))
moving = normalize_image(read_image(base['moving']))
fixed_sl = mid_slice(fixed, axis=0)
lo, hi = _bounds(fixed.shape, 'rigid')
span = hi - lo
particles = 8
progress_frames = 16
iterations = progress_frames
rng = np.random.default_rng(123 + pair_index)
positions = rng.uniform(lo, hi, size=(particles, lo.size)).astype(np.float32)
velocities = rng.uniform(-0.05 * span, 0.05 * span, size=positions.shape).astype(np.float32)
positions[0] = _identity_vector(fixed.ndim, 'rigid')
velocities[0] = 0.0
personal_best = positions.copy()
personal_scores = np.full(particles, np.inf, dtype=np.float32)
global_best = positions[0].copy()
global_score = float('inf')
frames = []
pso_progress_volumes = []
for iteration in range(iterations):
    for idx in range(particles):
        score = _score(fixed, moving, positions[idx], 'rigid', 'ncc')
        if score < personal_scores[idx]:
            personal_scores[idx] = score
            personal_best[idx] = positions[idx].copy()
        if score < global_score:
            global_score = float(score)
            global_best = positions[idx].copy()
    params = _vector_to_params(global_best, 'rigid', fixed.ndim)
    registered = normalize_image(_warp_rigid_3d(moving, params, output_shape=fixed.shape))
    pso_progress_volumes.append(registered)
    registered_sl = mid_slice(registered, axis=0)
    title = f'PSO true iteration | pair {pair_index:03d} | progress {iteration + 1:02d}/{iterations} | best score {global_score:.4f}'
    frames.append(registration_frame(fixed_sl, registered_sl, title, 'Best registered so far'))
    r1 = rng.random(size=positions.shape, dtype=np.float32)
    r2 = rng.random(size=positions.shape, dtype=np.float32)
    velocities = 0.72 * velocities + 1.45 * r1 * (personal_best - positions) + 1.45 * r2 * (global_best[None, :] - positions)
    positions = np.clip(positions + velocities, lo, hi)
save_gif(frames, out / f'pair_{pair_index:03d}_pso_true_iterations.gif', duration=240)


In [ ]:
# GIF type 5B: aligned progress comparison cho cả 4 methods, cùng 16 frame.
# PSO là iteration thật. Classical rerun với iteration tăng dần.
# VoxelMorph/TransMorph không có inference iteration, nên dùng final-field progress proxy.
out = gif_dir / '05_true_iterations'
cache = gif_dir / 'cache_classical_increasing_iters'
pair_index = demo_pair
base = row_for(pair_index, 'classical')
fixed = normalize_image(read_image(base['fixed']))
moving = normalize_image(read_image(base['moving']))
fixed_sl = mid_slice(fixed, axis=0)
progress_frames = 16
steps = list(range(progress_frames))
frames = []
classical_progress_volumes = []
for step in steps:
    if step == 0:
        registered = moving
    else:
        run_out = cache / f'pair_{pair_index:03d}_iter_{step:02d}'
        log = run_classical(base['fixed'], base['moving'], run_out, iterations=step, smoothing_sigma=1.3)
        registered = normalize_image(read_image(log['outputs']['registered']))
    classical_progress_volumes.append(registered)
    registered_sl = mid_slice(registered, axis=0)
    title = f'Classical Demons increasing iterations | pair {pair_index:03d} | progress {step + 1:02d}/{progress_frames} | iterations={step}'
    frames.append(registration_frame(fixed_sl, registered_sl, title, 'Registered'))
save_gif(frames, out / f'pair_{pair_index:03d}_classical_increasing_iterations.gif', duration=240)

progress_volumes = {
    'classical': classical_progress_volumes,
    'pso': pso_progress_volumes,
}
for method in ['voxelmorph', 'transmorph']:
    field = np.load(benchmark_dir / 'test_runs' / method / f'pair_{pair_index:03d}' / 'deformation_field.npy')
    progress_volumes[method] = [
        normalize_image(warp_dense_scaled(moving, field, method, alpha))
        for alpha in np.linspace(0.0, 1.0, progress_frames)
    ]
    method_frames = []
    for idx, volume in enumerate(progress_volumes[method]):
        title = f'{method_labels[method]} final-field progress proxy | pair {pair_index:03d} | progress {idx + 1:02d}/{progress_frames}'
        method_frames.append(registration_frame(fixed_sl, mid_slice(volume, axis=0), title, 'Warped by scaled final field'))
    save_gif(method_frames, out / f'pair_{pair_index:03d}_{method}_progress_proxy.gif', duration=240)

combined_frames = []
for idx in range(progress_frames):
    fig, axes = plt.subplots(2, 4, figsize=(13.5, 6.3))
    fig.suptitle(f'Aligned progress comparison | pair {pair_index:03d} | frame {idx + 1:02d}/{progress_frames}', fontsize=13)
    for col, method in enumerate(method_order):
        volume = progress_volumes[method][idx]
        candidate_sl = mid_slice(volume, axis=0)
        diff = np.abs(as_display(fixed_sl) - as_display(candidate_sl))
        axes[0, col].imshow(overlay_rgb(fixed_sl, candidate_sl))
        axes[0, col].set_title(method_labels[method], fontsize=10)
        axes[1, col].imshow(diff, cmap='magma')
        axes[1, col].set_title('|fixed - candidate|', fontsize=9)
        for ax in axes[:, col]:
            ax.set_xticks([])
            ax.set_yticks([])
    combined_frames.append(fig_to_frame(fig, dpi=115))
save_gif(combined_frames, out / f'pair_{pair_index:03d}_all_methods_aligned_progress.gif', duration=240)
